# Week 11: Contractual vs. Non-Contractual — Model Selection Framework
## Shifted Beta-Geometric (sBG) Model

**Masterclass:** [Week 11 Deep-Dive](https://balaviswanath-analytics.github.io/analytics_masterclass/week11_contractual_noncontractual.html)

**Dataset:** CDNOW (bundled) for non-contractual + simulated subscription data for contractual

**What You'll Build:**
1. Fader's 2x2 classification framework
2. sBG model for contractual settings (retention curve fitting)
3. BG/NBD vs. sBG: when to use which
4. Model class selection decision tree

In [ ]:
!pip install lifetimes pandas matplotlib -q

In [ ]:
import pandas as pd
import numpy as np
from scipy.optimize import minimize
from scipy.special import beta as beta_fn, betaln
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False
print("Libraries loaded.")

---
## Part 1: Fader's 2x2 Framework

| | Continuous | Discrete |
|---|---|---|
| **Contractual** (churn observed) | SaaS monthly → sBG, Cox PH | Annual membership → BdW, sBG |
| **Non-Contractual** (churn unobserved) | E-commerce → BG/NBD, Pareto/NBD | Seasonal → BG/BB |

**The key question:** "Do I know when a customer has churned?" If yes → contractual. If no → non-contractual.

---
## Part 2: Shifted Beta-Geometric (sBG) Model

For contractual settings with discrete renewal periods. Each customer has a personal churn probability theta drawn from a Beta distribution. Two parameters (a, b) capture the entire population heterogeneity.

In [ ]:
# Simulated subscription retention data (cohort of 10,000 subscribers)
np.random.seed(42)
n_customers = 10000
n_periods = 12

# True Beta distribution parameters
true_a, true_b = 0.7, 3.5

# Each customer draws a personal churn probability
theta = np.random.beta(true_a, true_b, n_customers)

# Simulate retention
retained = np.ones(n_customers, dtype=bool)
observed_retention = [1.0]  # period 0 = 100%
survivors_per_period = [n_customers]

for t in range(1, n_periods + 1):
    churns = np.random.rand(n_customers) < theta
    retained = retained & ~churns
    rate = retained.sum() / n_customers
    observed_retention.append(rate)
    survivors_per_period.append(retained.sum())

print("=== Observed Retention Rates ===")
for t, r in enumerate(observed_retention):
    print(f"  Period {t}: {r:.3f} ({int(r*n_customers):,} customers)")

In [ ]:
# === Fit sBG Model ===
# Survival function: S(t) = Beta(a, b+t) / Beta(a, b)
# Log-likelihood: sum of log-survivors and log-churners per period

retention_rates = np.array(observed_retention[1:])  # periods 1..12
n_start = n_customers

def sbg_log_likelihood(params, retention_rates, n_start):
    a, b = params
    if a <= 0 or b <= 0:
        return 1e10
    ll = 0
    for t in range(len(retention_rates)):
        # S(t+1) = Beta(a, b+t+1) / Beta(a, b)
        surv = np.exp(betaln(a, b + t + 1) - betaln(a, b))
        if t == 0:
            churn_prob = 1 - surv
        else:
            surv_prev = np.exp(betaln(a, b + t) - betaln(a, b))
            churn_prob = surv_prev - surv
        if churn_prob <= 0:
            churn_prob = 1e-10
        if surv <= 0:
            surv = 1e-10
        n_survived = int(retention_rates[t] * n_start)
        n_churned = int((retention_rates[max(0,t-1)] if t > 0 else 1.0) * n_start) - n_survived
        ll += n_churned * np.log(churn_prob) + n_survived * np.log(surv)
    return -ll

result = minimize(sbg_log_likelihood, x0=[1, 1], args=(retention_rates, n_start),
                  method='Nelder-Mead')
est_a, est_b = result.x

print(f"=== sBG Parameter Estimates ===")
print(f"  True:      a={true_a:.3f}, b={true_b:.3f}")
print(f"  Estimated: a={est_a:.3f}, b={est_b:.3f}")

# Predicted retention
predicted = [np.exp(betaln(est_a, est_b + t) - betaln(est_a, est_b))
             for t in range(1, n_periods + 2)]

fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(range(1, n_periods + 1), observed_retention[1:], 'o-', color='#264653',
        linewidth=2, markersize=8, label='Observed')
ax.plot(range(1, n_periods + 2), predicted, 's--', color='#c45d3e',
        linewidth=2, markersize=6, label='sBG Predicted')
ax.set_title('sBG Model: Observed vs Predicted Retention', fontsize=14, fontweight='bold')
ax.set_xlabel('Renewal Period')
ax.set_ylabel('Retention Rate')
ax.legend()
ax.set_ylim(0, 1)
plt.tight_layout()
plt.show()

In [ ]:
# Expected lifetime and CLV
E_lifetime = 1 + est_b / (est_a - 1) if est_a > 1 else float('inf')
avg_revenue_per_period = 50  # $/month
discount_rate = 0.01

# Discounted CLV
dcf_clv = sum(predicted[t] * avg_revenue_per_period / (1 + discount_rate)**(t+1)
              for t in range(len(predicted)))

print(f"Expected customer lifetime: {E_lifetime:.1f} periods")
print(f"Discounted CLV (12-period): ${dcf_clv:.2f}")

---
## Exercises

### Exercise 1: Model Selection Decision
You have an e-commerce business with no subscriptions. Customers buy irregularly. Which model class should you use? Why?

### Exercise 2: sBG vs. Exponential
Fit a simple exponential retention model (constant churn rate) to the same data. Compare fit quality to sBG. Why does sBG win?

### Exercise 3: Forecast Period 13-24
Use the fitted sBG to predict retention for months 13-24. At what month does retention drop below 10%?

---
## Key Takeaways
1. **Contractual vs. non-contractual** is the #1 model selection decision
2. **sBG** captures heterogeneity in churn probability with just 2 parameters
3. **Wrong model class** = months of wasted work (BG/NBD on subscription data is meaningless)
4. **Retention curve shape** reveals whether you have a heterogeneity problem